# 🚀 DL-PM25-BJ — Pipeline Huấn luyện GPU, Fine-tuning 12 Trạm & Ablation trên Kaggle

- **Dự án:** Dự báo nồng độ bụi mịn $\text{PM}_{2.5}$ tại Bắc Kinh (`beijing-pm25-deeplearning`)
- **Nhánh Git chuẩn:** `fix/BPM25_T0305_wp03-training-audit`
- **Tệp cấu hình duy nhất:** `configs/default.yaml`
- **Thư mục Backup Google Drive:** `https://drive.google.com/drive/u/0/folders/1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`
- **Google Drive Folder ID:** `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`

## Cell 1 — Kiểm tra Kaggle GPU (P100 / T4 x2) & Cấu hình Backup Drive

Thiết lập `GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"`, thư mục làm việc `/kaggle/working/course_project` và biến môi trường `MLFLOW_TRACKING_URI`.

In [1]:
import os
from pathlib import Path
import torch

GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"
KAGGLE_WORK_DIR = Path("/kaggle/working/course_project")
KAGGLE_BACKUP_DIR = Path("/kaggle/working/backups")
KAGGLE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] GDRIVE_FOLDER_ID = {GDRIVE_FOLDER_ID}")
print(f"[INFO] KAGGLE_WORK_DIR = {KAGGLE_WORK_DIR}")
print(f"[GPU] CUDA available: {torch.cuda.is_available()} | Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f"[GPU {idx}] {torch.cuda.get_device_name(idx)}")
!nvidia-smi

[INFO] GDRIVE_FOLDER_ID = 1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL
[INFO] KAGGLE_WORK_DIR = /kaggle/working/course_project
[GPU] CUDA available: True | Count: 2
[GPU 0] Tesla T4
[GPU 1] Tesla T4
Sat Nov 28 05:09:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W | 

## Cell 2 — Nạp Mã nguồn & Dữ liệu (Tầng 2 & Tầng 3)

- Hỗ trợ nạp mã nguồn từ `/kaggle/input/<dataset>` hoặc `git clone` nhánh `fix/BPM25_T0305_wp03-training-audit`.
- Nạp bộ 12 trạm từ `/kaggle/input/` (nếu đã gắn Kaggle Dataset) hoặc tải bằng `scripts/download_data.py`.
- Chạy `python -m src.data_pipeline.preprocess --config configs/default.yaml` tạo dữ liệu chuẩn 31 đặc trưng không Data Leakage.

In [5]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/Duy-DS/Deep_Learning_Group.git"  # Cập nhật URL repo thực tế
BRANCH_NAME = "fix/BPM25_T0305_wp03-training-audit"
KAGGLE_INPUT_CODE = Path("/kaggle/input/beijing-pm25-deeplearning/course_project")

if not KAGGLE_WORK_DIR.exists():
    if KAGGLE_INPUT_CODE.exists():
        shutil.copytree(KAGGLE_INPUT_CODE, KAGGLE_WORK_DIR)
        print(f"[INFO] Đã copy mã nguồn từ {KAGGLE_INPUT_CODE}")
    else:
        !git clone -b {BRANCH_NAME} {REPO_URL} /kaggle/working/repo
        if Path("/kaggle/working/repo/course_project").exists():
            shutil.move("/kaggle/working/repo/course_project", str(KAGGLE_WORK_DIR))
        else:
            shutil.move("/kaggle/working/repo", str(KAGGLE_WORK_DIR))

os.chdir(KAGGLE_WORK_DIR)
os.environ["MLFLOW_TRACKING_URI"] = str(KAGGLE_WORK_DIR / "mlruns")
print(f"[CWD] {os.getcwd()}")
print(f"[MLFLOW] MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

!grep -v '^torch' requirements.txt > /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt mlflow google-api-python-client google-auth google-auth-oauthlib

# Nạp dữ liệu thô từ Kaggle Input hoặc download_data.py
raw_dir = Path("data/raw")
if not list(raw_dir.glob("PRSA_Data_*.csv")):
    kaggle_csvs = list(Path("/kaggle/input").rglob("PRSA_Data_*.csv"))
    if kaggle_csvs:
        raw_dir.mkdir(parents=True, exist_ok=True)
        for csv_p in kaggle_csvs:
            shutil.copy2(csv_p, raw_dir / csv_p.name)
        print(f"[INFO] Đã nạp {len(kaggle_csvs)} tệp CSV từ /kaggle/input")
    else:
        !python scripts/download_data.py --output-dir data/raw

!python -m src.data_pipeline.preprocess --config configs/default.yaml
!ls -lh data/preprocess/

[CWD] /kaggle/working/course_project
[MLFLOW] MLFLOW_TRACKING_URI = /kaggle/working/course_project/mlruns
[DOWNLOADED] UCI dataset 501 | archive_bytes=8192212
[SAVED] /kaggle/working/course_project/data/raw/PRSA_Data_Aotizhongxin_20130301-20170228.csv | rows=35064 | SHA-256=a94fbcfc71708b6ffdc033360163d91efb7f675e496f6b5860929aa96b96351b
[SAVED] /kaggle/working/course_project/data/raw/PRSA_Data_Changping_20130301-20170228.csv | rows=35064 | SHA-256=a2e21402a6d95fb7043f87959e6e3dcbecb957aac356a6f8028a85b047c772e7
[SAVED] /kaggle/working/course_project/data/raw/PRSA_Data_Dingling_20130301-20170228.csv | rows=35064 | SHA-256=3d8dff81f18a80d9f455876a785642bffe1e9b9924ec1e6b7ce8cca790432eeb
[SAVED] /kaggle/working/course_project/data/raw/PRSA_Data_Dongsi_20130301-20170228.csv | rows=35064 | SHA-256=9c0020214425e5ee2a64d42f97a1f632e1d8f6b53ad50edd85a686cc7999f27e
[SAVED] /kaggle/working/course_project/data/raw/PRSA_Data_Guanyuan_20130301-20170228.csv | rows=35064 | SHA-256=59390fee3b032597b3

## Cell 3 — Huấn luyện Pre-train, Fine-tuning 12 Trạm & Ablation (3 Seeds `[42, 1024, 2026]`)

Thực thi trọn bộ pipeline từ `configs/default.yaml`:
1. Pre-training 4 mô hình (`lstm`, `dlinear`, `informer`, `patchtst`) trên các mốc $H \in \{24, 48, 96, 192\}$ với 3 seeds `[42, 1024, 2026]`.
2. Fine-tuning cho 12 trạm riêng lẻ (`fine_tune.py`).
3. Chạy ma trận Ablation Studies (`run_ablation.py`).

In [6]:
HORIZONS = [24, 48, 96, 192]

for h in HORIZONS:
    print(f"\n==================== PRE-TRAIN H={h} ====================")
    !python training_loop.py --config configs/default.yaml --horizon {h} --models lstm dlinear informer patchtst --seeds 42 1024 2026

for h in HORIZONS:
    print(f"\n==================== FINE-TUNE 12 TRẠM H={h} ====================")
    !python fine_tune.py --config configs/default.yaml --horizon {h} --models lstm dlinear informer patchtst --seed 42

print("\n==================== ABLATION STUDIES ====================")
!python run_ablation.py --config configs/default.yaml


==================== PRE-TRAIN H=24 ====================
Thiết bị: cuda | Mode: multivariate (D=31) | L=96, H=24 | Stride=4 | Loss=PeakShapeLoss | ValSel=balanced_dev | ResAnchor=False | Seeds: [42, 1024, 2026]
Số cửa sổ dữ liệu 4 tập: {'train': 74244, 'train_dev': 33420, 'dev': 33996, 'test': 50700} (var_train_dev=0.8735, var_dev=0.2619)

HUẤN LUYỆN MÔ HÌNH: LSTM
[LSTM | seed=42] Epoch 01/25 | train_loss=1.22596 | train_dev_mse=0.81631 | dev_mse=0.27088 | val_score=0.98433 | lr=8.06e-05
[LSTM | seed=42] Epoch 02/25 | train_loss=0.82379 | train_dev_mse=0.69547 | dev_mse=0.24584 | val_score=0.86736 | lr=1.60e-04
[LSTM | seed=42] Epoch 03/25 | train_loss=0.63661 | train_dev_mse=0.65497 | dev_mse=0.26942 | val_score=0.88920 | lr=2.40e-04
[LSTM | seed=42] Epoch 04/25 | train_loss=0.53564 | train_dev_mse=0.68117 | dev_mse=0.25430 | val_score=0.87532 | lr=3.00e-04
[LSTM | seed=42] Epoch 05/25 | train_loss=0.46267 | train_dev_mse=0.73666 | dev_mse=0.22981 | val_score=0.86034 | lr=2.97e-04
[L

## Cell 4 — Xuất CSV MLflow & Đóng gói/Upload lên Google Drive (`1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`)

1. Xuất `mlflow_runs_export.csv` chuẩn sheet `Raw_MLflow_Logs` và đóng gói `dl_pm25_bj_backup_[YYMMDD]_[HHMM].zip` tại `/kaggle/working/`.
2. Tự động upload trực tiếp vào Google Drive Folder ID `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL` thông qua **Kaggle Secrets**:
   - Ưu tiên 1: Secret `GDRIVE_SERVICE_ACCOUNT_JSON` (nội dung JSON của Service Account đã được share quyền Editor vào thư mục `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`).
   - Ưu tiên 2: Secret `GDRIVE_OAUTH_TOKEN` (OAuth2 Access Token hợp lệ).
3. Luôn giữ lại bản `.zip` và `.csv` trong `/kaggle/working/` làm phương án tải thủ công dự phòng.

In [9]:
import json
import shutil
from pathlib import Path

!python -m src.utils.exp_backup --mlruns-dir mlruns --output-dir outputs --backup-dir /kaggle/working/backups --include-checkpoints

latest_zip = sorted(KAGGLE_BACKUP_DIR.glob("dl_pm25_bj_backup_*.zip"))[-1]
csv_export = Path("outputs/mlflow_runs_export.csv")
if csv_export.exists():
    shutil.copy2(csv_export, Path("/kaggle/working") / csv_export.name)
shutil.copy2(latest_zip, Path("/kaggle/working") / latest_zip.name)
print(f"[LOCAL BACKUP] Đã lưu bản dự phòng tại /kaggle/working/{latest_zip.name}")

try:
    from kaggle_secrets import UserSecretsClient
    from google.oauth2 import service_account
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    secrets = UserSecretsClient()
    creds = None
    try:
        sa_raw = secrets.get_secret("GDRIVE_SERVICE_ACCOUNT_JSON")
        creds = service_account.Credentials.from_service_account_info(
            json.loads(sa_raw),
            scopes=["https://www.googleapis.com/auth/drive.file"],
        )
    except Exception:
        try:
            oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN")
            creds = Credentials(token=oauth_token)
        except Exception:
            creds = None

    if creds is None:
        print("[WARN] Chưa cấu hình Kaggle Secret 'GDRIVE_SERVICE_ACCOUNT_JSON' hoặc 'GDRIVE_OAUTH_TOKEN'. Dùng bản tải thủ công tại /kaggle/working/.")
    else:
        drive_service = build("drive", "v3", credentials=creds)
        for upload_file, mime in [(latest_zip, "application/zip"), (csv_export, "text/csv")]:
            if not upload_file.exists():
                continue
            metadata = {"name": upload_file.name, "parents": [GDRIVE_FOLDER_ID]}
            media = MediaFileUpload(str(upload_file), mimetype=mime, resumable=True)
            res = drive_service.files().create(body=metadata, media_body=media, fields="id").execute()
            print(f"[GDRIVE API] Đã upload {upload_file.name} -> file_id={res.get('id')} vào folder {GDRIVE_FOLDER_ID}")
except Exception as exc:
    print(f"[WARN] Bỏ qua upload Google Drive ({exc}). Bản backup vẫn sẵn sàng tại /kaggle/working/{latest_zip.name}.")

<frozen runpy>:130: RuntimeWarning: 'src.utils.exp_backup' found in sys.modules after import of package 'src.utils', but prior to execution of 'src.utils.exp_backup'; this may result in unpredictable behaviour
[EXPORTED] Đã xuất 0 MLflow runs ra: /kaggle/working/course_project/outputs/mlflow_runs_export.csv
[BACKUP] Đã đóng gói 582 tệp tại: /kaggle/working/backups/dl_pm25_bj_backup_261128_0712.zip
[LOCAL BACKUP] Đã lưu bản dự phòng tại /kaggle/working/dl_pm25_bj_backup_261128_0712.zip
[WARN] Chưa cấu hình Kaggle Secret 'GDRIVE_SERVICE_ACCOUNT_JSON' hoặc 'GDRIVE_OAUTH_TOKEN'. Dùng bản tải thủ công tại /kaggle/working/.


In [1]:
# Vá nhanh lỗi thiếu --lookback trong run_ablation.py ngay trên Kaggle
with open("run_ablation.py", "r", encoding="utf-8") as f:
    code = f.read()

if '"--lookback"' not in code and "'--lookback'" not in code:
    code = code.replace(
        'parser.add_argument("--config"',
        'parser.add_argument("--lookback", type=int, default=96)\n    parser.add_argument("--config"'
    )
    with open("run_ablation.py", "w", encoding="utf-8") as f:
        f.write(code)
    print("Đã vá thành công --lookback vào run_ablation.py!")

FileNotFoundError: [Errno 2] No such file or directory: 'run_ablation.py'